# CSPB-3104 Programming Assignment 9



1) (5 points) Implement Kruskal's algorithm 

Input: An edge list with weights: [(0,1,1), (0,2,2),(1,2,1)]  
Output: A minimum spanning tree in the form of an edge list with weights: [(0, 1, 1), (1, 2, 1)] 

Note: Edge lists are lists of triples (i, j, w), with i < j, which represents an edge between nodes i and j with weight w.  Edges are undirected in this notebook, and you should always return edges in the form (i, j, w), where i < j. Make sure to sort your final edge list in natural order, ie (0, 2, 1) before (1,2,1), (0,1,0) before (0,2,0).

Hint: Look into Python's Set class


In [46]:
from scipy.cluster.hierarchy import DisjointSet

def kruskal(a):
    forest = DisjointSet({})
    total_nodes = len(forest)
    for n1, n2, weight in a:
        forest.add(n1)
        forest.add(n2)

    mst = []
    sorted_edges = sorted(a, key=lambda x: (x[2], x[0], x[1]))
    for n1, n2, weight in sorted_edges:
        if (len(mst) == total_nodes - 1):
            break

        if (forest.connected(n1, n2)):
            continue
        else:
            forest.merge(n1, n2)
            mst.append((n1, n2, weight))
    
    sorted_mst = sorted(mst, key=lambda x: (x[0], x[1], x[2]))

    return sorted_mst   



2. (5 points) Implement Prim's algorithm

Input: An edge list with weights: [(0,1,1), (0,2,2),(1,2,1)]  
Output: A minimum spanning tree in the form of an edge list with weights: [(0, 1, 1), (1, 2, 1)] 

Note: Edge lists are lists of triples (i, j, w), with i < j, which represents an edge between nodes i and j with weight w.  Edges are undirected in this notebook, and you should always return edges in the form (i, j, w), where i < j. Make sure to sort your final edge list in natural order, ie (0, 2, 1) before (1,2,1), (0,1,0) before (0,2,0).

Hint: You can use heapq for the priority queue.

In [53]:
import heapq

def prim(a):
    nodes = set()
    adj_list = {}
    for n1, n2, weight in a:
        nodes.add(n1)
        adj_list.setdefault(n1, []).append((n2, weight))
        nodes.add(n2)
        adj_list.setdefault(n2, []).append((n1, weight))
    
    edge_heap = []
    for n2, weight in adj_list[0]:
        new_edge = (weight, 0, n2)
        heapq.heappush(edge_heap, new_edge)

    seen = set([0])
    mst = []

    while edge_heap and len(seen) < len(nodes):
        edge = heapq.heappop(edge_heap)
        weight = edge[0]
        n1 = edge[1]
        n2 = edge[2]

        if n2 in seen:
            continue
        else:
            seen.add(n2)

        mst.append((n1, n2, weight))

        for n3, new_weight in adj_list[n2]:
            if n3 in seen:
                continue
            else:
                new_edge = (new_weight, n2, n3)
                heapq.heappush(edge_heap, new_edge)

        for edge in range(len(mst)):
            n1 = mst[edge][0]
            n2 = mst[edge][1]
            weight = mst[edge][2]
            if n1 > n2:
                swapped = (n2, n1, weight)
                mst[edge] = swapped

    sorted_mst = sorted(mst, key=lambda x: (x[0], x[1], x[2]))

    return sorted_mst   



3) (15 points)  Finding the most likely mutation tree

You're given a list of bacteria RNA fragments, all from related bacteria which have mutated into separate strains over time.  Your goal is to come up with the most likely sequence of mutations that led to this state of affairs.  

The chance that one bacteria mutated into another depends on the number of differences in their RNA strings. 
The more differences in their RNA strings, the more unlikely it is that the bacteria mutated into each other.  (In fact, exponentially more unlikely -- the probability that k locations changed at the same time is $2^{-k}$).

If we construct a fully connected graph whose nodes represent RNA fragments and each edge has weight $2^{-k}$, where k is the number of differences between RNA strings, then a spanning tree which *maximizes* the *product* of edge weights will be the __most likely mutation tree__.  (Each mututation is assumed to be independent, so the chance that all the mutations in the spanning tree happen is the product of their respective probabilities)

Write a function that takes a list of RNA fragments, constructs an edge list with weights, then returns the most likely mutation tree, along with its probability.  

Note: your algorithm should construct a graph and then run your implementation of Kruskal's algorithm on it.  The difficulty lies in determining the correct graph, so that a minimum sum spanning tree in your graph corresponds to a maximum product spanning tree in the graph described above.

Input: ["adad","adac","acad", "cdac","addd"]  
Output: ([('adad', 'adac', 0.5),
  ('adad', 'acad', 0.5),
  ('adad', 'addd', 0.5),
  ('adac', 'cdac', 0.5)],
 0.0625)

In [48]:
# Given a list of RNA fragments, returns a spanning tree which maximizes the probability of mutation
from scipy.cluster.hierarchy import DisjointSet

def kruskal_max(a):
    forest = DisjointSet({})
    total_nodes = len(forest)
    for n1, n2, weight in a:
        forest.add(n1)
        forest.add(n2)

    mst = []
    sorted_edges = sorted(a, key=lambda x: (x[2], x[0], x[1]), reverse=True)
    for n1, n2, weight in sorted_edges:
        if (len(mst) == total_nodes - 1):
            break

        if (forest.connected(n1, n2)):
            continue
        else:
            forest.merge(n1, n2)
            mst.append((n1, n2, weight))
    
    sorted_mst = sorted(mst, key=lambda x: (x[0], x[1], x[2]))

    return sorted_mst

def mutation_tree(a):
    edges = []

    for i in range(len(a)):
        for j in range(i + 1, len(a)):
            f1 = a[i]
            f2 = a[j]
            differences = len(f1) - sum(a == b for a, b in zip(f1, f2))
            chance = 2 ** (-differences)
            edges.append((f1, f2, chance))
    
    mst = kruskal_max(edges)
    probability = 1

    for n1, n2, weight in mst:
        probability *= weight
    
    mst_sorted = sorted(mst, key=lambda x: a.index(x[0]))
    return (mst_sorted, probability)


Testing below

----

In [49]:
## DO NOT EDIT TESTING CODE FOR YOUR ANSWER ABOVE
# Press shift enter to test your code. Ensure that your code has been saved first by pressing shift+enter on the previous cell.
from IPython.display import display, HTML
def kruskal_test():
    failed = False
    test_cases = [ 
        ([(0,1,1), (0,2,2),(1,2,1)], [(0, 1, 1), (1, 2, 1)]),
        ([(0,1,2), (0,4,1), (1,2,1), (1,4,2), (2,3,1), (3,4,1)], 
         [(0, 4, 1), (1, 2, 1), (2, 3, 1), (3, 4, 1)]),
        ([(0,1,1), (0,2,2), (0,3,1), (1,4,1), (1,5,2), (2,4,2), 
          (2,6,2), (3,5,2), (3,6,1), (4,7,2), (5,7,2), (6,7,1)], 
          [(0, 1, 1), (0, 2, 2), (0, 3, 1), (1, 4, 1), (1, 5, 2), (3, 6, 1), (6, 7, 1)]),
        ([(0,1,2), (0,2,2), (0,3,1), (1,4,1), (1,5,1), (2,4,2), 
          (2,6,1), (3,5,2), (3,6,2), (4,7,2), (5,7,2), (6,7,1)], 
         [(0, 1, 2), (0, 2, 2), (0, 3, 1), (1, 4, 1), (1, 5, 1), (2, 6, 1), (6, 7, 1)]) 
    ]
    for (test_graph, solution) in test_cases:
        output = kruskal(test_graph)
        if (solution != output):
            s1 = '<font color=\"red\"> Failed - test case: Inputs: graph =' + str(test_graph) + "<br>"
            s2 = '  <b> Expected Output: </b> ' + str(solution) + ' Your code output: ' + str(output)+ "<br>"
            display(HTML(s1+s2))
            failed = True
            
    if failed:
        display(HTML('<font color="red"> One or more tests failed. </font>'))
    else:
        display(HTML('<font color="green"> All tests succeeded! </font>'))
kruskal_test()

In [54]:
## DO NOT EDIT TESTING CODE FOR YOUR ANSWER ABOVE
# Press shift enter to test your code. Ensure that your code has been saved first by pressing shift+enter on the previous cell.
from IPython.display import display, HTML
def prim_test():
    failed = False
    test_cases = [ 
        ([(0,1,1), (0,2,2),(1,2,1)], [(0, 1, 1), (1, 2, 1)]),
        ([(0,1,2), (0,4,1), (1,2,1), (1,4,2), (2,3,1), (3,4,1)], 
         [(0, 4, 1), (1, 2, 1), (2, 3, 1), (3, 4, 1)]),
        ([(0,1,1), (0,2,2), (0,3,1), (1,4,1), (1,5,2), (2,4,2), 
          (2,6,2), (3,5,2), (3,6,1), (4,7,2), (5,7,2), (6,7,1)], 
          [(0, 1, 1), (0, 2, 2), (0, 3, 1), (1, 4, 1), (1, 5, 2), (3, 6, 1), (6, 7, 1)]),
        ([(0,1,2), (0,2,2), (0,3,1), (1,4,1), (1,5,1), (2,4,2), 
          (2,6,1), (3,5,2), (3,6,2), (4,7,2), (5,7,2), (6,7,1)], 
         [(0, 1, 2), (0, 2, 2), (0, 3, 1), (1, 4, 1), (1, 5, 1), (2, 6, 1), (6, 7, 1)]) 
    ]
    for (test_graph, solution) in test_cases:
        output = prim(test_graph)
        if (solution != output):
            s1 = '<font color=\"red\"> Failed - test case: Inputs: graph =' + str(test_graph) + "<br>"
            s2 = '  <b> Expected Output: </b> ' + str(solution) + ' Your code output: ' + str(output)+ "<br>"
            display(HTML(s1+s2))
            failed = True
            
    if failed:
        display(HTML('<font color="red"> One or more tests failed. </font>'))
    else:
        display(HTML('<font color="green"> All tests succeeded! </font>'))
prim_test()

In [ ]:
## DO NOT EDIT TESTING CODE FOR YOUR ANSWER ABOVE
# Press shift enter to test your code. Ensure that your code has been saved first by pressing shift+enter on the previous cell.
from IPython.display import display, HTML
def mutation_test():
    failed = False
    test_cases = [ 
        (["TAT", "CAT", "CAC"],([('TAT', 'CAT', 0.5), ('CAT', 'CAC', 0.5)], 0.25)),
        (["ACATA", "ATCTA", "GTCTA", "GTATA", "GCATA"], 
        ([('ACATA', 'GCATA', 0.5), ('ATCTA', 'GTCTA', 0.5), ('GTCTA', 'GTATA', 0.5), ('GTATA', 'GCATA', 0.5)], 0.0625)),
        (["GATTACA", "CGACTCA", "CATTACA", "CGACATA", "CGTTACA", "CGACACA", "CATTACG", "CGATACA"], 
         ([('GATTACA', 'CATTACA', 0.5), ('CGACTCA', 'CGACACA', 0.5), ('CATTACA', 'CGTTACA', 0.5), 
           ('CATTACA', 'CATTACG', 0.5), ('CGACATA', 'CGACACA', 0.5), ('CGTTACA', 'CGATACA', 0.5), ('CGACACA', 'CGATACA', 0.5)], 0.0078125)),
        (["CATTACA", "GATTACA", "CTTTACA", "CTGGTGA", "CTGTACA", "CTGGTCA", "CTGGTGC", "CTGGACA"], 
        ([('CATTACA', 'GATTACA', 0.5), ('CATTACA', 'CTTTACA', 0.5), ('CTTTACA', 'CTGTACA', 0.5), 
          ('CTGGTGA', 'CTGGTCA', 0.5), ('CTGGTGA', 'CTGGTGC', 0.5), ('CTGTACA', 'CTGGACA', 0.5), ('CTGGTCA', 'CTGGACA', 0.5)], 0.0078125))
    ]
    for (test_graph, solution) in test_cases:
        output = mutation_tree(test_graph)
        if (solution != output):
            s1 = '<font color=\"red\"> Failed - test case: Inputs: graph =' + str(test_graph) + "<br>"
            s2 = '  <b> Expected Output: </b> ' + str(solution) + ' Your code output: ' + str(output)+ "<br>"
            display(HTML(s1+s2))
            failed = True
            
    if failed:
        display(HTML('<font color="red"> One or more tests failed. </font>'))
    else:
        display(HTML('<font color="green"> All tests succeeded! </font>'))
mutation_test()